# Lab 03｜文件在磁盘上，模型就已经知道了吗？

这是专栏《自进化 Agent 实战》03 讲《上下文引擎：模型每轮对话都会看见什么？》的配套练习。练习使用构造的故障调查材料，目标是掌握处理真实场景的能力。

## 题面

值班助手已收到支付服务的延迟告警，详细部署记录也放在它有权限访问的工作目录里，它却一直说“不知道最近是否改过连接池”。同事建议把“认真检查所有资料”写进提示词，另一个人则认为工具事件里出现过文件名，就说明模型已经读过正文。你会检查什么来判断这份资料到底在哪一步没有被用上？如果助手随后读取了这份记录，下一次判断为什么能得到新的信息？请说清一次模型请求是怎样准备出来的，以及工作目录、任务历史和这次请求之间的区别。

## 参考答案

我会先查这份记录是否真的进入了关键判断之前的那次模型请求。文件在工作目录里，只说明工具有机会读到它；事件里出现过文件名，也可能只是列目录时扫到的。两者都不能证明模型已经收到正文。只加一句“认真检查”，改变的是模型选择动作的倾向，却不能把还没读的文件自动变成已有事实。

每次调用模型时，运行环境会把这一轮要交给模型的内容整理好：基础指令、本轮允许使用的工具定义，以及会话里保留下来的消息；开发者附加的要求也在消息里，不混同于一段普通提示词。工具说明只告诉模型能做什么，真正提供“这次读到了什么”的是工具回执。

关键是这一轮的内容不是任务开始时定死、一路用到结束的。模型提出读取动作后，运行环境执行工具，把返回写进历史；只要任务还要继续，下一次调用开始前，会根据更新后的历史重新准备输入。文件正文、完整任务记录和当前这一轮实际发送的内容，区别就在这里：前两者保存着可用材料与发生过的事，当前这一轮只装着这次真正发出去的东西。

排查时，我会找成功读取的命令和它的回执，再检查后续请求里的工具结果项，而不是只看回答像不像知道了变更。练习把随机采集编号只写在部署文件里：读文件前的请求没有这个编号，读取后的工具结果应当带着它；再看它从哪一次请求开始进入后续判断。


## 准备

使用 Python 3.12 内核。需要 `GLM_API_KEY`，未设置时会弹出隐藏输入框；依赖自动补装。`lab.codex` 是真实的 `Codex` 客户端。下面把固定指令直接传给 `thread_start()`，再通过 `thread.turn(...).run()` 观察三轮调用。


In [ ]:
import sys
from pathlib import Path
examples = next(p for base in (Path.cwd(), *Path.cwd().parents)
                for p in (base, base / "examples") if (p / "notebook03_setup.py").is_file())
sys.path.insert(0, str(examples))
from notebook03_setup import prepare
story = prepare()


## 文件就在目录里，Agent 为什么没有用上？

先看构造的告警与部署文件，在同一会话先只依据告警回答，再读取部署记录后回答同一个问题。
跑完对照“同一问题”和“本次实际回答”，看已确认的事实与待补的信息怎样变化。
对应专栏“文件就在目录里，Agent 为什么没有用上？”；部署文件中的时间、参数与随机编号均为构造值。


In [ ]:
from IPython.display import Markdown, display
from openai_codex import ApprovalMode, Codex, Sandbox
lab, work, marker = story.lab, story.work, story.marker
rule, alert, question = story.rule, story.alert, story.question
display(Markdown("### deployment.md（构造材料）\n```text\n" + story.deployment.read_text(encoding="utf-8") + "\n```"))
codex: Codex = lab.codex if lab else None
thread = codex.thread_start(cwd=str(work), developer_instructions=rule,
    sandbox=Sandbox.full_access, approval_mode=ApprovalMode.deny_all) if codex else None
before_start = len(lab.requests) if lab else 0
before = None
if lab:
    turn = thread.turn(alert + "\n" + question + "\n不要读取文件或调用工具。", sandbox=Sandbox.full_access)
    with lab.watch(turn):
        before = turn.run()
    lab.record_result(turn, before, show_result=False)


In [ ]:
read_start = len(lab.requests) if lab else 0
result = None
if lab:
    turn = thread.turn("先读取 deployment.md，再回答：" + question
        + "不要查询其他文件或网络；工具失败后停止，不重试。", sandbox=Sandbox.full_access)
    with lab.watch(turn):
        result = turn.run()
    lab.record_result(turn, result, show_result=False)
read_end = len(lab.requests) if lab else 0
story.show_answers(before, result, before_start, read_start, read_end, thread)


## 模型这一轮看见的，不是整个工作目录

打开告警轮与读取轮的请求，核对基础指令、工具定义、任务消息和开发者规则分别在哪里。
跑完对照请求组成与编号附近的真实内容，找随机编号首次进入哪次请求的工具结果。
对应专栏“模型这一轮看见的，不是整个工作目录”；成功回执与请求内容分别核对，不靠回答语气判断。


In [ ]:
from notebook_support import command_receipts, tool_output_contains
observed_calls = lab.requests[before_start:read_end] if lab else []
read_calls = lab.requests[read_start:read_end] if lab else []
receipts = command_receipts(read_calls)
marker_requests = [(i, tool_output_contains(request, marker))
                   for i, request in enumerate(observed_calls, before_start + 1)]
request_parts = story.request_parts(observed_calls[0] if observed_calls else {})


In [ ]:
first_marker_request = next((i for i, entered in marker_requests if entered), None)
print("部署编号首次进入工具结果的请求序号：", first_marker_request if lab else "离线没有模型请求")
story.show_requests(observed_calls, before_start)
story.show_composition(request_parts, receipts, marker_requests)


## 谁来准备模型每一轮看到的内容？

在同一会话追问采集编号与采集时间，不重述记录，也不要求再次读取文件。
跑完对照旧工具结果是否仍在输入中、本轮是否新增工具调用，以及输入文件是否改变。
对应专栏“谁来准备模型每一轮看到的内容？”；下一次输入根据更新的历史重新准备。


In [ ]:
followup_start = len(lab.requests) if lab else 0
followup = None
if lab:
    turn = thread.turn("根据刚才已经取得的记录，回答采集编号和采集时间。不要再调用工具。", sandbox=Sandbox.full_access)
    with lab.watch(turn):
        followup = turn.run()
    lab.record_result(turn, followup, show_result=False)
followup_calls = lab.requests[followup_start:] if lab else []
previous_ids = {item.get("call_id") for request in observed_calls for item in request.get("input", [])
                if isinstance(item, dict) and item.get("type") == "function_call"}
new_tool_calls = story.new_tool_calls(followup_calls, previous_ids, followup)
after = story.snapshot(work)


In [ ]:
print("输入文件：", "未改变" if after == story.input_snapshot else "有变化")
print("追问新增工具调用：", len(new_tool_calls) if lab else "离线没有调用")
story.show_followup(followup, followup_calls, read_calls, new_tool_calls, after, thread)
